# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

The FAIR^2 dataset package contains clinical and pathological features of colorectal cancer survivors. All entities (record sets, fields, columns) are referenced using their `@id` fields in this notebook.

In [ ]:
# Ensure mlcroissant is installed!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"\nDataset title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

### Record Sets

The dataset may have one or more record sets containing relevant records. We'll list all record set `@id`s and their key field `@id`s.

In [ ]:
# List all record sets and their fields by @id
record_sets = dataset.list_record_sets()
print("Available record sets and their field @ids:")
rs_fields = {}
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}   (name: {rs.get('name', 'N/A')})")
    fields = dataset.list_fields(record_set=rs['@id'])
    rs_fields[rs['@id']] = [field['@id'] for field in fields]
    for field in fields:
        print(f"    - Field: {field['@id']}   (name: {field.get('name','N/A')})  dataType: {field.get('dataType','N/A')}")

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames for analysis.

All extraction uses `@id` references for record sets and fields.

In [ ]:
# Load all record sets into DataFrames
dataframes = {}
print("\nExtracting records...\n")

record_set_ids = list(rs_fields.keys())
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    df = pd.DataFrame(records)
    dataframes[rsid] = df
    print(f"Loaded DataFrame for RecordSet @id: {rsid} with shape {df.shape}")

# Preview columns for first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Columns in DataFrame for RecordSet @id '{main_record_set_id}':")
    print(dataframes[main_record_set_id].columns.tolist())
    # Show the head
    dataframes[main_record_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply typical data processing steps: filtering, normalization, and grouping.

We will:
- Filter records by a numeric field (e.g., Age)
- Normalize values
- Group by a key variable (e.g., Sex)

All fields are referenced by their `@id`.

In [ ]:
# Choose a numeric field for EDA
# Find a numeric field from the first record set
main_fields = rs_fields.get(main_record_set_id, [])
numeric_field_id = None
group_field_id = None
fields_info = dataset.list_fields(record_set=main_record_set_id) if main_record_set_id else []
for f in fields_info:
    if f.get('dataType') in ['schema:Integer', 'schema:Float', 'schema:Number']:
        numeric_field_id = f['@id']
    if f.get('name','').lower() in ['sex', 'gender'] or f.get('dataType') == 'schema:Text':
        group_field_id = f['@id']
if not numeric_field_id:
    print("No numeric field found; please choose manually.")
else:
    print(f"Numeric field for EDA: {numeric_field_id}")
if not group_field_id:
    print("No suitable group field found; please choose manually.")

# Continue if we have a numeric field
if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    # Make sure column exists
    if numeric_field_id in df.columns:
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 10
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records where {numeric_field_id} > {threshold:.2f}")
            print(filtered_df.head())

            # Normalize
            filtered_df[numeric_field_id + '_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"\nNormalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

            # Group by key attribute
            if group_field_id and group_field_id in df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"\nGrouped by {group_field_id} (mean of {numeric_field_id}):")
                print(grouped_df)
            else:
                print("Grouping field unavailable or not present in dataframe.")
        except Exception as e:
            print(f"Error processing field {numeric_field_id}: {e}")
    else:
        print(f"Field {numeric_field_id} not found in columns for record set {main_record_set_id}.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. All column references use their `@id`s.

In [ ]:
# Example: Visualize numeric distribution and group comparison
if main_record_set_id and numeric_field_id:
    df = dataframes[main_record_set_id]
    plt.figure(figsize=(8,5))
    if numeric_field_id in df.columns:
        sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
        plt.title(f"Distribution of field '{numeric_field_id}'")
        plt.xlabel(numeric_field_id)
        plt.ylabel("Count")
        plt.show()
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step exploration of the FAIR^2 dataset using `mlcroissant`:
- Dataset loaded via its Croissant schema URL
- Record sets, fields, and columns referenced by their `@id`
- Extraction and transformation of tabular records
- Simple EDA and visualization by key clinical attributes

Further analysis can focus on clinicopathological predictors for MSI-H phenotype, anatomical distribution, and comorbidity patterns in CRC survivors, leveraging FAIR^2's robust schema metadata.